# Lab 5: Distributed Deep Learning

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This lecture is part of the course [Machine Learning for Earth Observation powered by Supercomputers (REI506M, 2023)](https://ugla.hi.is/kennsluskra/index.php?tab=nam&chapter=namskeid&id=71151920236&kennsluar=2023)

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 4 | Understanding Transformers | ✅ Previous |
| Lab 4.1 | Training on Sentinel-2 Data | ✅ Previous |
| Lab 5 | **Distributed Training (Multi-GPU)** | 🔄 **Current** |
| Lab 6 | Validation & Performance Metrics | ⬜ Next |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Final |

---

## Project Context

**Coming From Lab 4.1:**
- You've trained a transformer on 1-2 GPUs
- Training took several hours
- Now you want to scale to 4+ GPUs for faster iteration

**What You'll Do:**
- Set up PyTorch DDP for multi-GPU training
- Write Slurm scripts for multi-node jobs
- Train the same model 4-8x faster
- Save checkpoints for Lab 6

**Output:**
- Trained model on distributed infrastructure
- Faster experimentation for hyperparameter tuning

---

**Outline**

[TOC]

## Lecture Content
The lecture will delve into the utilization of PyTorch Lightning with Distributed Data Parallel (DDP) for optimizing the training of a Transformer model on remote sensing data. It will include code demonstrations on setting up DDP in PyTorch Lightning and effectively inputting remote sensing data into a Transformer architecture.

## Learning Objectives
* Become familiar with batch scripts to launch multi-node jobs on HPC systems.
* Understand the methods of PyTorch DDP to scale DL models on multiple GPUs.

## Why Use PyTorch DDP for Distributed Deep Learning?
**Improved Scalability**: DDP enables your model to scale across multiple GPUs and nodes, significantly improving training times on large datasets.

**Efficient Utilization of Resources**: By distributing the model and data across multiple GPUs, DDP ensures efficient usage of available computational resources.

**Reduced Communication Overhead**: DDP minimizes the communication overhead by synchronizing gradients locally and only exchanging them across nodes when necessary.

**Flexibility and Control**: DDP provides fine-grained control over the distribution process, allowing for more customized and optimized distributed training setups.

**Easy Integration**: DDP can be integrated into existing PyTorch models with minimal changes to the codebase, making it accessible for most deep learning projects.

### Adapting Code to Use PyTorch DDP
**Initialization**: Start by initializing the process group with `torch.distributed.init_process_group`, which sets up the communication between the nodes.

In [ ]:
import torch.distributed as dist
dist.init_process_group(backend='nccl', init_method='env://')

**Model Wrapping**: Wrap your model with `torch.nn.parallel.DistributedDataParallel`. This wrapper takes care of gradient averaging and synchronization.

In [ ]:
model = torch.nn.parallel.DistributedDataParallel(model)

**Data Loading**: Modify your data loader to use `torch.utils.data.distributed.DistributedSampler`. This sampler ensures that each process receives a unique subset of the dataset, avoiding data duplication.

In [ ]:
train_sampler = torch.utils.data.distributed.DistributedSampler(dataset)
train_loader = torch.utils.data.DataLoader(dataset, sampler=train_sampler)

**Running the Training Loop**: The training loop largely remains the same, but make sure to set the sampler's epoch at the beginning of each epoch.

In [ ]:
for epoch in range(num_epochs):
    train_sampler.set_epoch(epoch)
    # training loop

### Using PyTorch DDP with PyTorch Lightning
PyTorch Lightning simplifies the process further by abstracting much of the boilerplate code required for distributed training.

**Configuration**: In PyTorch Lightning, you only need to specify the number of GPUs and the type of distributed backend in the Trainer class.

In [ ]:
from pytorch_lightning import Trainer
trainer = Trainer(gpus=4, distributed_backend='ddp')

**Model Definition**: Define your model as a subclass of `pytorch_lightning.LightningModule`. Implement the necessary methods (`training_step`, `configure_optimizers`, etc.).

**Data Module**: Use a LightningDataModule to define your data loading logic. This module is automatically compatible with distributed training.

**Training**: Simply pass your model and data module to the trainer's fit method. Lightning handles the rest.

In [ ]:
trainer.fit(model, data_module)

By following these steps, you can effectively adapt your deep learning models for distributed training using PyTorch DDP and PyTorch Lightning, leading to more efficient and scalable model training processes.

### Code
We are going to use code from Lab 4:
https://gitlab.jsc.fz-juelich.de/hedgedoc/gVineQJMQ_-XjJHq4k9yhg
[PyTorch Lightning introduction](https://lightning.ai/docs/pytorch/stable/starter/introduction.html).

---

## Summary

This lab covered:

1. **PyTorch DDP**: Distributed Data Parallel for multi-GPU training
2. **PyTorch Lightning Integration**: Simplified distributed training setup
3. **Slurm Job Scripts**: Submitting distributed jobs to HPC clusters
4. **Performance Scaling**: Training on 8+ GPUs across multiple nodes
5. **Checkpoint Management**: Saving models for inference and evaluation

### Performance Expectations

| Setup | Time for 150 Epochs | Speedup |
|-------|-------------------|---------|
| 1 GPU | 6 hours | 1x |
| 2 GPUs | 3.5 hours | 1.7x |
| 4 GPUs | 2 hours | 3x |
| 8 GPUs (2 nodes) | 1 hour | 6x |

---

## Next Steps

### After Training Completes

1. **Collect Best Checkpoint**
   ```bash
   # Copy model to shared location
   cp checkpoints/best_model.ckpt ~/models/lab5_transformer_ddp.ckpt
   ```

2. **Save Training Metrics**
   ```bash
   # Lightning creates tensorboard logs
   tensorboard --logdir lightning_logs/
   ```

3. **Prepare for Lab 6**
   - Your best checkpoint is ready for inference
   - Make note of validation loss achieved
   - Document hyperparameters used

### Lab 6: Validation & Evaluation

Next, you'll:
- Load the trained model from this lab
- Run inference on validation data
- Calculate accuracy metrics:
  - Overall Accuracy (OA)
  - Producer's Accuracy (PA) per class
  - User's Accuracy (UA) per class
  - Confusion matrices
- Compare with other land cover products

### Example Lab 6 Code Preview

```python
# Load checkpoint from Lab 5
model = TransformerModel.load_from_checkpoint("lab5_transformer_ddp.ckpt")

# Run inference
predictions = trainer.predict(model, val_loader)

# Calculate metrics
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_true, y_pred)
oa = accuracy_score(y_true, y_pred)
```

---

## Troubleshooting & FAQ

**Q: How many nodes should I use?**
- Start with 2-4 nodes (8-16 GPUs) for faster training
- Beyond that, communication overhead increases
- Batch size needs to scale with GPU count

**Q: Can I resume interrupted training?**
- Yes! PyTorch Lightning auto-resumes from last checkpoint
- Use `trainer = Trainer(..., resume_from_checkpoint="path_to_checkpoint")`

**Q: How does DDP scale beyond 8 GPUs?**
- Use Ring AllReduce for gradient synchronization
- Efficient up to ~32 GPUs
- Beyond that, consider model parallelism

**Q: What's the difference between DDP and DataParallel?**
- **DataParallel**: Single process, multiple GPUs (slower, easier to debug)
- **DDP**: Multiple processes, multiple GPUs (faster, production-ready)
- Always use DDP for distributed training

---

## References

- [PyTorch DDP Documentation](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html)
- [PyTorch Lightning DDP Tutorial](https://lightning.ai/docs/pytorch/stable/advanced/model_parallel/ddp.html)
- [NCCL Backend Documentation](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/)

---

**Continue to Lab 6 for validation →**